# MSD MedSAM3: matched controls and constraints, three folds × three seeds

**No augmentation. Original fine-tuning recipe: 200 optimizer updates, accumulation 2,
native SAM3 loss, medical pretrained LoRA, 10% auxiliary-gradient target and 50% p95 cap.**
This notebook is **not capped at 31.98 TFLOPs** and is not compute-matched to the four-model MSD/ADNI sweeps.

It retains folds **0, 1, 3** and seeds **17, 83, 191**, with five original support volumes per fold.
**126 fresh runs** cover all 14 established conditions. Fresh baseline, bands and bands+edge
controls run first. The 27 historical cluster results are retained for reference only; they are
never used as matched controls or counted as completed runs in this study.

All coefficients, including bands and edge, are calibrated sequentially on this runtime's
training support, once per fold and shared across seeds. The original 10% median target is
limited by the original 50% p95 rule; the median may be lower when the p95 cap binds.
SDF uses full-volume foreground-class signed distances clipped at 8 mm; depth uses the existing
normalized anatomical depth CE on the native 1-mm RAS grid. All active auxiliary terms use the
original 20-update ramp. There is no new SDF delay, validation tuning or early stopping.

Use an A100 or H100 with at least 40 GB GPU RAM and a High-RAM runtime. GPU preflight is mandatory.
All primary comparisons use fresh Colab controls in the same GPU/software environment.
These selected folds support exploratory comparisons, not unbiased full CV. Production CUDA
kernels retain the original numerical variability; equal seeds do not guarantee bitwise replay.


## 1. Upload and connect Drive
Create `MyDrive/hippo_medsam3_msd_extension/input` and upload these three files:
`medsam3_msd_matched_code.zip`, `medsam3_msd_data.zip`, and `sam3.pt`.
**If the data ZIP and SAM3 checkpoint are already there, upload only the new code ZIP.**
The MedSAM3 medical adapter is included in the code ZIP. Keep the original files on Drive.
This uses a different folder from the running MSD and ADNI notebooks.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, shutil, signal, subprocess, sys, zipfile
HOME_DIR = Path('/content/drive/MyDrive/hippo_medsam3_msd_extension')
INPUT = HOME_DIR/'input'
RESULTS = HOME_DIR/'results_matched200_noaug_v2'
BUNDLE = Path('/content/hippo_medsam3_msd_matched_bundle')
PREVIOUS_RESULTS = HOME_DIR/'results_original200_noaug_v1'
PREVIOUS_BUNDLE = Path('/content/hippo_medsam3_msd_bundle')
RESULTS.mkdir(parents=True, exist_ok=True)


## 2. Verify uploads and stage on the VM SSD
Copies the 3.45 GB frozen SAM3 checkpoint once. The dataset is the original NIfTI collection,
not the resized PKL used by the four-model sweep. Upload and extraction hashes are checked.
This uses separate bundle and results folders and leaves the earlier study intact.
On the same VM it reuses the verified local base checkpoint through a hard link; otherwise
it copies the existing checkpoint from Drive. No new data transfer is required.


In [ ]:
EXPECTED_UPLOADS = {'medsam3_msd_matched_code.zip': '1a6a0130083c90632f3ba392c67e9bb9940908e84ff4f4d6cfb86714f61d7239', 'medsam3_msd_data.zip': 'ad869cf8c8862697b41bda740e91cae91bea6fcfdadca364b6f3ec5e6acf95e4', 'sam3.pt': '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e'}
EXPECTED_MANIFEST = '39788851ab86a8df615031a2f6fe460be09cc045c666944fb4ebe74481b74654'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024**2), b''): h.update(block)
    return h.hexdigest()
def checked_copy(source, target, expected):
    if target.exists() and sha256(target) == expected: return
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name+'.copying')
    shutil.copyfile(source, temporary)
    if sha256(temporary) != expected: raise ValueError('Copy failed verification: '+str(target))
    temporary.replace(target)
def safe_extract(path, destination):
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            name = Path(item.filename)
            if name.is_absolute() or '..' in name.parts or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Unsafe ZIP member')
        archive.extractall(destination)
for name, expected in EXPECTED_UPLOADS.items():
    path = INPUT/name
    if not path.exists(): raise FileNotFoundError('Upload '+name+' into '+str(INPUT))
    if sha256(path) != expected: raise ValueError('Upload hash mismatch: '+name)
    print('Verified', name, flush=True)
if shutil.disk_usage('/content').free < 16*1024**3:
    raise RuntimeError('Need 16 GiB free on the runtime disk before staging')
if (BUNDLE/'bundle_manifest.json').exists() and sha256(BUNDLE/'bundle_manifest.json') != EXPECTED_MANIFEST:
    raise RuntimeError('Existing matched bundle differs; preserve its work before replacing it.')
BUNDLE.mkdir(exist_ok=True)
safe_extract(INPUT/'medsam3_msd_matched_code.zip', BUNDLE)
safe_extract(INPUT/'medsam3_msd_data.zip', BUNDLE)
base = BUNDLE/'weights/sam3.pt'
previous_base = PREVIOUS_BUNDLE/'weights/sam3.pt'
if not base.exists() and previous_base.exists() and sha256(previous_base) == EXPECTED_UPLOADS['sam3.pt']:
    os.link(previous_base, base)
checked_copy(INPUT/'sam3.pt', base, EXPECTED_UPLOADS['sam3.pt'])
if sha256(BUNDLE/'bundle_manifest.json') != EXPECTED_MANIFEST: raise ValueError('Bundle manifest differs')
(RESULTS/'source_binding.json').write_text(json.dumps(dict(uploads=EXPECTED_UPLOADS, manifest=EXPECTED_MANIFEST), indent=2))
print('Staged on the local SSD. No training has started.')


## 3. Create the environment
Uses the original dependency versions. Installation output is captured and displayed on error.
The pip bootstrap handles Colab runtimes without ensurepip. No cluster connection is needed.


In [ ]:
VENV = Path('/content/hippo_medsam3_venv')
PYTHON = VENV/'bin/python'
def setup(args, log_name):
    result = subprocess.run(list(map(str,args)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (RESULTS/log_name).write_text(result.stdout)
    print('\n'.join(result.stdout.splitlines()[-35:]))
    if result.returncode: raise RuntimeError(f'Command failed; full output: {RESULTS/log_name}')
if not PYTHON.exists():
    setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV], 'venv_setup.log')
probe = subprocess.run([str(PYTHON),'-m','pip','--version'], capture_output=True, text=True)
if probe.returncode:
    import urllib.request
    bootstrap = Path('/content/medsam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', bootstrap)
    setup([PYTHON,bootstrap], 'pip_bootstrap.log')
setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'], 'dependencies.log')
def campaign(command, *extra):
    args = [str(PYTHON),str(BUNDLE/'runtime.py'),command,'--bundle',str(BUNDLE),'--durable',str(RESULTS),*map(str,extra)]
    process = subprocess.Popen(args, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_path = RESULTS/(command+'.log')
    try:
        with log_path.open('a') as log:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line);log.flush()
        code = process.wait()
    except KeyboardInterrupt:
        print('Requesting clean stop. Wait for recovery upload to finish.')
        os.killpg(process.pid, signal.SIGINT);process.wait();return
    if code: raise RuntimeError(f'{command} failed. Details are printed above and saved in {log_path}')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## 4. Preserve and reuse completed preflight work
This reads the previous failed-preflight folder without moving or deleting it. Complete saved
measurements and passed audits are reused only after checking the exact previous package,
GPU/software, unchanged code, data, medical adapter and all audit checks. It imports no weights
or training results. Fold 0 and fold 1 can therefore reuse their 20 measurements each on the
same A100 runtime. A different environment runs all preflight work anew. A manual STOP is preserved.
This cell does not approve the new campaign; run the required preflight below.


In [ ]:
campaign('reuse-preflight', '--previous-results', PREVIOUS_RESULTS, '--previous-bundle', PREVIOUS_BUNDLE)


## 5. Required GPU preflight and shared calibration
Verifies exact pretrained parameter hashes, original data/calibration context, native/bands/edge
loss and gradient equivalence, unchanged label-free inference, and exact optimizer/RNG resume.
Strict numerical comparisons run in a separate process with deterministic GPU algorithms,
including an original-versus-original repeatability control. Original BF16 precision is retained.
Only in that audit, the empty point-prompt sampler uses its exact zero derivative while retaining
native forward execution. Nonempty sampling remains subject to the original strict checks.
Calibration and training retain the original runtime settings; strict diagnostic settings do not
carry over. Exact checkpoint replay is tested under the diagnostic settings, while production GPU
kernels can have the same run-to-run numerical variation observed in the original implementation.
Then calibrates every term from 20 training slabs per fold; no historical coefficients are reused. No scientific training updates
are charged during these diagnostics. It may take substantially longer than a simple smoke test.
Completed fold receipts are reused only for the same bundle, GPU and software.


In [ ]:
campaign('preflight')


## 6. Run or resume the 126 matched experiments
One experiment at a time; already archived runs in this study are skipped. The local checkpoint is saved every update;
a coherent recovery snapshot is uploaded to Drive asynchronously about every three minutes.
Each update is recorded before execution. A hard VM loss with newer charged updates than the
durable checkpoint blocks automatic replay and retains recovery for an audit.

Interrupt once for a clean stop and wait for the upload. `SESSION_HOURS` is a conservative
stop target, not a runtime guarantee. A `STOP` file in RESULTS also requests a clean stop;
remove it deliberately when ready to resume. Do not run two copies against this results folder.

Dice, FN/FP, A/P swaps, training curves, configuration, calibration and source identities are
archived and hash-verified after each run. Its adapter/optimizer recovery files are then retired.
Evaluation uses the final 200-update checkpoint, with no validation-based selection.


In [ ]:
MAX_NEW_RUNS = 126
SESSION_HOURS = 22.0
campaign('run', '--max-runs', MAX_NEW_RUNS, '--hours', SESSION_HOURS)


## 7. Compare with fresh matched controls
Dice and gains are averaged across seeds within each fold, then equally across the three folds.
Gains require the fresh baseline with the same fold and seed. Missing baselines are never filled
from the historical cluster study. The 27 old results remain in `historical_reference` for context.
The seed-46 pilot is excluded. Training curves and final train/validation metrics are retained;
no new per-epoch validation schedule is introduced.


In [ ]:
campaign('report')
import pandas as pd
comparison = json.loads((RESULTS/'comparison.json').read_text())
display(pd.DataFrame(comparison['aggregate']))
display(pd.DataFrame(comparison['runs']))
